In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Dataset with Class Indices for Targets
# Target classes: 0 = Working, 1 = Further Studying
X = torch.tensor([
    [3.8, 4.0],  # Working
    [2.1, 1.0],  # Further studying
    [3.5, 3.0],  # Working
    [2.5, 2.0],  # Further studying
    [3.9, 4.0],  # Working
    [2.0, 4.0],  # Further studying
], dtype=torch.float32)

# Notice y is now class indices (integers: 0 or 1), not floats [1.0] or [0.0]
y = torch.tensor([0, 1, 0, 1, 0, 1], dtype=torch.long)

# 2. Updated Neural Network Architecture
class StudentPercentageNetwork(nn.Module):
    def __init__(self):
        super(StudentPercentageNetwork, self).__init__()
        self.hidden1 = nn.Linear(2, 2)
        self.relu1 = nn.ReLU()

        self.hidden2 = nn.Linear(2, 2)
        self.relu2 = nn.ReLU()

        # CHANGED: Output layer now has 2 nodes (1 for Working, 1 for Further Study)
        self.output_layer = nn.Linear(2, 2)

        # Note: We omit Softmax here because PyTorch's CrossEntropyLoss
        # expects raw model outputs (logits) for numerical stability.
        # We apply Softmax manually only when we want to view the percentages.
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.hidden1(x)
        x = self.relu1(x)
        x = self.hidden2(x)
        x = self.relu2(x)
        x = self.output_layer(x)  # Returns raw scores for both choices
        return x

# Initialize model
model = StudentPercentageNetwork()

# 3. CHANGED: Use CrossEntropyLoss for multi-class percentage distribution
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

# 4. Train the Model
epochs = 500
print("--- Training Started ---")
for epoch in range(epochs):
    predictions = model(X)  # Raw logits
    loss = criterion(predictions, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

print("--- Training Completed ---")

# 5. Test/Predict with a New Student Profile (GPA = 3.6, Year = 4.0)
test_student = torch.tensor([[3.6, 4.0]], dtype=torch.float32)

# Pass through model and apply Softmax to get percentages
raw_output = model(test_student)
probabilities = nn.Softmax(dim=1)(raw_output)

prob_working = probabilities[0][0].item()
prob_further = probabilities[0][1].item()

print(f"\nPrediction for GPA 3.6, Year 4:")
print(f" - Working: {prob_working * 100:.2f}%")
print(f" - Further Study: {prob_further * 100:.2f}%")

--- Training Started ---
Epoch [100/500], Loss: 0.6931
Epoch [200/500], Loss: 0.6931
Epoch [300/500], Loss: 0.6931
Epoch [400/500], Loss: 0.6931
Epoch [500/500], Loss: 0.6931
--- Training Completed ---

Prediction for GPA 3.6, Year 4:
 - Working: 50.00%
 - Further Study: 50.00%
